# Notebook 1 — Fix Cache Rebuild

**Purpose:** Rebuild the MoE embedding cache using the fixed supervision pipeline.

## What changed (vs the broken Sprint 3 cache)
| # | Bug fixed | Impact |
|---|---|---|
| RC1 | `prior_mask` fallback no longer uses whole lung | Experts stop painting the entire lung |
| RC2 | `_ringify_mask` removed from supervision | Cavity expert gets learnable targets |
| RC4 | Weight floor removed; low-confidence experts get weight=0.05 | Expert specialisation restored |
| RC5 | TBX11K `ActiveTuberculosis`/`ObsoletePulmonaryTuberculosis` now map to correct experts | TBX bbox GT actually used |
| RC6 | `mabdullahi454/tb-pipeline-checkpoints` added to candidate paths | C1 LoRA + C4 decoder loaded correctly |

## Datasets to attach on Kaggle
| Dataset slug | Mount path |
|---|---|
| `iahmedhabib/medsam-vit-b` | `/kaggle/input/datasets/iahmedhabib/medsam-vit-b/` |
| `mabdullahi454/tb-pipeline-checkpoints` | `/kaggle/input/datasets/mabdullahi454/tb-pipeline-checkpoints/` |
| `iahmedhabib/montgomery` | `/kaggle/input/datasets/iahmedhabib/montgomery/montgomery` |
| `iahmedhabib/shehzhenn` | `/kaggle/input/datasets/iahmedhabib/shehzhenn/shenzhen` |
| `usmanshams/tbx-11` | `/kaggle/input/datasets/usmanshams/tbx-11/TBX11K` |
| `organizations/nih-chest-xrays` | `/kaggle/input/datasets/organizations/nih-chest-xrays/data` |

**GPU:** T4 × 1 (required — MedSAM encoder needs it)

**Estimated runtime:** ~40–50 min (1000 images × 4 datasets, ~0.6 s/image)

In [ ]:
# ── Cell 1: Clone repo and install dependencies ──────────────────────────────
import os, subprocess, sys

REPO_URL  = "https://github.com/AhmedHabib00/dl-project-codebase.git"
REPO_DIR  = "/kaggle/working/repo"
CACHE_DIR = "/kaggle/working/moe_cache_v2"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     os.path.join(REPO_DIR, "requirements.txt")],
    check=True
)
print("Setup complete.")

In [ ]:
# ── Cell 2: Verify checkpoint mounts ────────────────────────────────────────
from pathlib import Path

MEDSAM_CKPT = Path("/kaggle/input/datasets/iahmedhabib/medsam-vit-b/medsam_vit_b.pth")
C1_ADAPTER  = Path("/kaggle/input/datasets/mabdullahi454/tb-pipeline-checkpoints/component1_adapters.safetensors")
C4_DECODER  = Path("/kaggle/input/datasets/mabdullahi454/tb-pipeline-checkpoints/component4_mask_decoder.pt")

MONTGOMERY  = Path("/kaggle/input/datasets/iahmedhabib/montgomery/montgomery")
SHENZHEN    = Path("/kaggle/input/datasets/iahmedhabib/shehzhenn/shenzhen")
TBX11K      = Path("/kaggle/input/datasets/usmanshams/tbx-11/TBX11K")
NIH         = Path("/kaggle/input/datasets/organizations/nih-chest-xrays/data")

def check(label, path, required=True):
    status = "OK" if path.exists() else "MISSING"
    flag   = "[REQUIRED]" if required else "[optional]"
    print(f"  {flag} {label:<40}: {status}  ({path})")
    if required and not path.exists():
        raise FileNotFoundError(f"{label} not found at {path}")

check("MedSAM checkpoint",  MEDSAM_CKPT)
check("C1 LoRA adapter",    C1_ADAPTER)
check("C4 lung decoder",    C4_DECODER)
check("Montgomery dataset", MONTGOMERY)
check("Shenzhen dataset",   SHENZHEN)
check("TBX11K dataset",     TBX11K)
check("NIH-CXR14 dataset",  NIH, required=False)
print("All required mounts present.")

In [ ]:
# ── Cell 3: Patch configs with absolute Kaggle paths ────────────────────────
import yaml

moe_yaml_path   = Path(REPO_DIR) / "configs" / "moe.yaml"
paths_yaml_path = Path(REPO_DIR) / "configs" / "paths.yaml"

with moe_yaml_path.open() as f:
    moe_cfg = yaml.safe_load(f)

moe_cfg["component1"]["checkpoint_path"]          = str(MEDSAM_CKPT)
moe_cfg["component1"]["adapter_path"]              = str(C1_ADAPTER)
moe_cfg["component4"]["checkpoint_path"]          = str(MEDSAM_CKPT)
moe_cfg["component4"]["decoder_checkpoint_path"]  = str(C4_DECODER)

with moe_yaml_path.open("w") as f:
    yaml.safe_dump(moe_cfg, f, sort_keys=False)

# Paths override
paths_cfg = {
    "project_root": REPO_DIR,
    "datasets": {
        "montgomery": str(MONTGOMERY),
        "shenzhen":   str(SHENZHEN),
        "tbx11k":     str(TBX11K),
        "nih_cxr14":  str(NIH) if NIH.exists() else "/kaggle/working/_missing_nih",
    },
}
with paths_yaml_path.open("w") as f:
    yaml.safe_dump(paths_cfg, f, sort_keys=False)

print("Configs patched.")
print(f"  C1 adapter : {moe_cfg['component1']['adapter_path']}")
print(f"  C4 decoder : {moe_cfg['component4']['decoder_checkpoint_path']}")

In [ ]:
# ── Cell 4: Smoke test — build cache for 8 images only (~3 min) ─────────────
# Run this cell before the full build to verify the pipeline runs end-to-end.
import subprocess, sys
from pathlib import Path

SMOKE_CACHE = "/kaggle/working/moe_cache_smoke"

cmd = [
    sys.executable, "-m", "scripts.cache_moe_embeddings",
    "--config",          str(Path(REPO_DIR) / "configs" / "moe.yaml"),
    "--paths",           str(Path(REPO_DIR) / "configs" / "paths.yaml"),
    "--output",          SMOKE_CACHE,
    "--limit-per-domain","8",
]
result = subprocess.run(cmd, cwd=REPO_DIR, capture_output=False)
assert result.returncode == 0, "Smoke cache build failed — check output above."
smoke_files = list(Path(SMOKE_CACHE).glob("*.pt"))
print(f"Smoke cache: {len(smoke_files)} files written to {SMOKE_CACHE}")

In [ ]:
# ── Cell 5: Sanity check — verify non-zero mask fractions (smoke cache) ─────
# EXPECTED:
#   consolidation : 25–45% non-empty
#   cavity        :  8–20% non-empty  (was 0–2% with ringify + whole-lung fallback)
#   fibrosis      : 15–35% non-empty
#   nodule        : 15–30% non-empty
#   NIH (if included): ALL FOUR experts should be close to 0%
# If NIH is > 10%, the prior-mask fallback fix did not take effect — stop.

import torch
from collections import defaultdict

expert_names = ["consolidation", "cavity", "fibrosis", "nodule"]
stats = {e: {"total": 0, "nonzero": 0} for e in expert_names}
nih_stats = {e: {"total": 0, "nonzero": 0} for e in expert_names}

for p in Path(SMOKE_CACHE).glob("*.pt"):
    d = torch.load(p, weights_only=False)
    is_nih = str(p.stem).startswith("nih")
    for e in expert_names:
        mask = d["expert_masks"][e]
        has_content = float(mask.sum().item()) > 0
        if is_nih:
            nih_stats[e]["total"]   += 1
            nih_stats[e]["nonzero"] += int(has_content)
        else:
            stats[e]["total"]   += 1
            stats[e]["nonzero"] += int(has_content)

print("Expert non-zero mask fractions (non-NIH samples):")
for e in expert_names:
    t = stats[e]["total"]
    n = stats[e]["nonzero"]
    pct = 100.0 * n / t if t else 0.0
    flag = "OK" if pct > 2 else "WARN"
    print(f"  [{flag}] {e:<16}: {n}/{t}  ({pct:.1f}%)")

print("\nNIH expert non-zero fractions (should all be ~0%):")
for e in expert_names:
    t = nih_stats[e]["total"]
    n = nih_stats[e]["nonzero"]
    pct = 100.0 * n / t if t else 0.0
    flag = "OK" if pct < 10 else "FAIL — prior-mask fix not active"
    print(f"  [{flag}] {e:<16}: {n}/{t}  ({pct:.1f}%)")

In [ ]:
# ── Cell 6: Full cache build — 1000 images per domain (~40–50 min) ──────────
# Only run after Cell 5 passes (NIH non-zero < 10%).

import subprocess, sys
from pathlib import Path

cmd = [
    sys.executable, "-m", "scripts.cache_moe_embeddings",
    "--config",          str(Path(REPO_DIR) / "configs" / "moe.yaml"),
    "--paths",           str(Path(REPO_DIR) / "configs" / "paths.yaml"),
    "--output",          CACHE_DIR,
    "--limit-per-domain","1000",
]
result = subprocess.run(cmd, cwd=REPO_DIR)
assert result.returncode == 0, "Full cache build failed — check output above."

cache_files = list(Path(CACHE_DIR).glob("*.pt"))
print(f"Full cache: {len(cache_files)} files written to {CACHE_DIR}")

In [ ]:
# ── Cell 7: Post-build stats — dataset balance and expert mask distribution ──
import torch
from collections import Counter
from pathlib import Path

expert_names = ["consolidation", "cavity", "fibrosis", "nodule"]
dataset_counts = Counter()
expert_nonzero = {e: Counter() for e in expert_names}
expert_total   = Counter()

for p in Path(CACHE_DIR).glob("*.pt"):
    dataset_id = p.stem.split("__")[0]
    dataset_counts[dataset_id] += 1
    d = torch.load(p, weights_only=False)
    for e in expert_names:
        expert_total[e] += 1
        if float(d["expert_masks"][e].sum().item()) > 0:
            expert_nonzero[e][dataset_id] += 1

print("Dataset sample counts (balanced sampler will equalise during training):")
for ds, cnt in sorted(dataset_counts.items()):
    print(f"  {ds:<20}: {cnt}")

print("\nExpert non-zero target fractions:")
total = sum(dataset_counts.values())
for e in expert_names:
    n = sum(expert_nonzero[e].values())
    pct = 100.0 * n / total if total else 0.0
    print(f"  {e:<16}: {n}/{total}  ({pct:.1f}%)")

print("\nAll good! Copy the CACHE_DIR path for use in Notebook 2:")
print(f"  CACHE_DIR = '{CACHE_DIR}'")